# 08 - Feature Importance & Selection

## Context
Dengan lebih dari 400 fitur hasil rekayasa (feature engineering), tidak semua fitur memberikan kontribusi yang berarti bagi prediksi model. Notebook ini bertujuan untuk melatih model baseline (LightGBM) guna mengukur tingkat kepentingan (importance) masing-masing fitur. Hasil ini akan membantu kita memilih fitur terbaik, mengurangi dimensi data, dan mempercepat waktu training pada tahap modeling lanjutan.

## Objective
- Melatih model LightGBM secara cepat untuk mengestimasi feature importance.
- Mengidentifikasi fitur yang memiliki kontribusi rendah atau nol.
- Memilih top N fitur dengan gain tertinggi untuk modeling selanjutnya.
- Menyimpan peringkat feature importance ke folder metadata.

### Impor Library & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

# Tentukan path data terproses
data_path = Path('../data/processed/train_features.parquet')
if not data_path.exists():
    data_path = Path('../data/interim/train_merged.parquet')

train = pd.read_parquet(data_path)
print(f'Data berhasil dimuat: {train.shape}')

### Persiapan Fitur & Split Data

In [ ]:
# Kolom yang tidak digunakan untuk modeling
exclude_cols = ['TransactionID', 'isFraud', 'TransactionDT']
object_cols = train.select_dtypes(include=['object']).columns.to_list()

# Pilih hanya fitur numerik
feature_cols = [c for c in train.columns if c not in exclude_cols and c not in object_cols]

X = train[feature_cols]
y = train['isFraud']

# Pembagian train/val menggunakan stratifikasi agar fraud rate seimbang
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Jumlah fitur yang dievaluasi: {len(feature_cols)}")
print(f"Distribusi target: {y.sum():,} fraud / {len(y):,} total ({y.mean()*100:.2f}%)")

### Pelatihan Model LightGBM untuk Feature Importance

In [ ]:
# Inisialisasi parameter model baseline
params = {
    'objective': 'binary',
    'metric': 'auc',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.1,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': -1,
    'n_jobs': -1,
    'random_state': 42
}

lgb_train = lgb.Dataset(X_train, label=y_train)
lgb_val = lgb.Dataset(X_val, label=y_val, reference=lgb_train)

# Latih model dengan early stopping
evals_result = {}
model = lgb.train(
    params,
    lgb_train,
    num_boost_round=200,
    valid_sets=[lgb_train, lgb_val],
    valid_names=['train', 'valid'],
    callbacks=[
        lgb.early_stopping(stopping_rounds=30),
        lgb.log_evaluation(period=20),
        lgb.record_evaluation(evals_result)
    ]
)

# Prediksi dan hitung skor AUC
y_pred = model.predict(X_val)
val_auc = roc_auc_score(y_val, y_pred)

print(f'\nIterasi terbaik: {model.best_iteration}')
print(f'Validation AUC: {val_auc:.4f}')

### Visualisasi: Learning Curve & ROC Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot kurva pembelajaran (learning curve)
axes[0].plot(evals_result['train']['auc'], label='Train AUC', linewidth=2)
axes[0].plot(evals_result['valid']['auc'], label='Valid AUC', linewidth=2)
axes[0].axvline(x=model.best_iteration, color='r', linestyle='--', label=f'Iterasi Terbaik ({model.best_iteration})')
axes[0].set_xlabel("Boosting Round")
axes[0].set_ylabel('AUC')
axes[0].set_title(f'Kurva Pembelajaran (Best AUC: {val_auc:.4f})')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot kurva ROC
fpr, tpr, _ = roc_curve(y_val, y_pred)
axes[1].plot(fpr, tpr, linewidth=2, label=f'LightGBM (AUC = {val_auc:.4f})')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].fill_between(fpr, tpr, alpha=0.2)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Kurva ROC')
axes[1].legend(loc='lower right')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

### Analisis: Hitung Nilai Feature Importance

In [ ]:
# Hitung tingkat kepentingan fitur berdasarkan metrik gain
importance = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print('Top 20 Fitur Paling Berpengaruh:')
print(importance.head(20).to_string(index=False))

### Visualisasi: Top 30 Fitur Paling Berpengaruh

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
top30 = importance.head(30)
ax.barh(range(30),
        top30['importance'].values,
        color=plt.cm.Blues(np.linspace(0.8, 0.2, 30)))

ax.set_yticks(range(30))
ax.set_yticklabels(top30['feature'].values)
ax.invert_yaxis()
ax.set_xlabel('Tingkat Kepentingan (Gain)')
ax.set_title('Top 30 Fitur Berdasarkan Nilai Gain')
plt.tight_layout()
plt.show()

### Seleksi Fitur & Simpan Hasil

In [ ]:
# Simpan seluruh daftar feature importance ke metadata
output_path = Path('../data/metadata')
output_path.mkdir(parents=True, exist_ok=True)
importance.to_csv(output_path / 'feature_importance.csv', index=False)

print(f'Peringkat feature importance berhasil disimpan ke: {output_path / "feature_importance.csv"}')